In [14]:
import geopandas as gpd
import rasterio
import pandas as pd

In [15]:
pop_tanzania = rasterio.open('../tza_pop_2026_CN_1km_R2025A_UA_v1.tif')

In [16]:
pd.DataFrame(pop_tanzania.dataset_mask().flatten())[0].unique()

array([  0, 255], dtype=uint8)

In [17]:
import numpy as np

# Read population values from band 1, masking NoData pixels.
population = pop_tanzania.read(1, masked=True)
rows, cols = np.where(~np.ma.getmaskarray(population))
x, y = rasterio.transform.xy(pop_tanzania.transform, rows, cols, offset="center")

# Coordinates use the TIFF's CRS (see pop_tanzania.crs).
population_df = pd.DataFrame({
    "x": x,
    "y": y,
    "population": population.data[rows, cols],
})

population_df.head()


,x,y,population
0,30.762499,-0.987500,142.387009
1,30.770832,-0.987500,18.808517
2,30.754166,-0.995833,40.088570
3,30.762499,-0.995833,95.127190
4,30.770832,-0.995833,25.027851


In [18]:
from pathlib import Path

from scipy import sparse
from sklearn.pipeline import Pipeline
from water_wells import (
    DataConfig, DataLoader, DataPreprocessor, FeatureEngineer, TrainingConfig,
)

# Support running Jupyter from either the project root or notebooks/.
project_root = Path.cwd()
if not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

# The loader reads the two root CSVs and aligns labels by well ID.
X_raw, y = DataLoader(DataConfig(data_dir=project_root)).load_training()
config = TrainingConfig()

# Reuse the same feature/preprocessing steps as TrainingPipeline, without a model.
preprocessing_pipeline = Pipeline([
    ("features", FeatureEngineer()),
    ("preprocessing", DataPreprocessor(config.min_frequency, config.max_categories)),
])
X_preprocessed = preprocessing_pipeline.fit_transform(X_raw, y)

# Readable features for filtering, plotting, and geographic exploration.
training_df = preprocessing_pipeline.named_steps["features"].transform(X_raw)
training_df.insert(0, "id", X_raw["id"])
training_df["status_group"] = y

# Fully preprocessed model features; sparse columns avoid a large dense copy.
feature_names = preprocessing_pipeline.named_steps["preprocessing"].get_feature_names_out()
if sparse.issparse(X_preprocessed):
    preprocessed_df = pd.DataFrame.sparse.from_spmatrix(
        X_preprocessed, index=X_raw.index, columns=feature_names,
    )
else:
    preprocessed_df = pd.DataFrame(
        X_preprocessed, index=X_raw.index, columns=feature_names,
    )
preprocessed_df.insert(0, "id", X_raw["id"])
preprocessed_df["status_group"] = y

# Full-data preprocessing is for exploration; split raw rows first for evaluation.
print(f"Readable: {training_df.shape}; preprocessed: {preprocessed_df.shape}")
training_df.head()


Readable: (59400, 45); preprocessed: (59400, 492)


,id,amount_tsh,funder,gps_height,installer,longitude,latitude,wpt_name,num_private,basin,...,source_class,waterpoint_type,waterpoint_type_group,nearby_wells_3km,nearby_wells_5km,nearby_wells_10km,nearby_wells_20km,well_age,construction_year_missing,status_group
0,69572,6000.0,Roman,1390,Roman,34.938093,-9.856322,none,0,Lake Nyasa,...,groundwater,communal standpipe,communal standpipe,18.0,18.0,29.0,91.0,12.0,0,functional
1,8776,0.0,Grumeti,1399,GRUMETI,34.698766,-2.147466,Zahanati,0,Lake Victoria,...,surface,communal standpipe,communal standpipe,26.0,28.0,29.0,31.0,3.0,0,functional
2,34310,25.0,Lottery Club,686,World vision,37.460664,-3.821329,Kwa Mahundi,0,Pangani,...,surface,communal standpipe multiple,communal standpipe,16.0,16.0,18.0,106.0,4.0,0,functional
3,67743,0.0,Unicef,263,UNICEF,38.486161,-11.155298,Zahanati Ya Nanyumbu,0,Ruvuma / Southern Coast,...,groundwater,communal standpipe multiple,communal standpipe,12.0,17.0,29.0,53.0,27.0,0,non functional
4,19728,0.0,Action In A,0,Artisan,31.130847,-1.825359,Shuleni,0,Lake Victoria,...,surface,communal standpipe,communal standpipe,8.0,18.0,47.0,156.0,NaN,1,functional


In [19]:
training_df.isna().sum()

id                               0
amount_tsh                       0
funder                        3637
gps_height                       0
installer                     3655
longitude                        0
latitude                         0
wpt_name                         0
num_private                      0
basin                            0
region                           0
region_code                      0
district_code                    0
lga                              0
ward                             0
population                       0
public_meeting                3334
recorded_by                      0
scheme_management             3878
scheme_name                  28810
permit                        3056
construction_year            20709
extraction_type                  0
extraction_type_group            0
extraction_type_class            0
management                       0
management_group                 0
payment                          0
payment_type        

In [20]:
preprocessed_df


,id,funder__funder_0,funder__funder_Adb,funder__funder_Adra,funder__funder_Amref,funder__funder_Ces(gmbh),funder__funder_Danida,funder__funder_Ded,funder__funder_Dh,funder__funder_Dhv,...,numerical__district_code,numerical__population,numerical__construction_year,numerical__nearby_wells_3km,numerical__nearby_wells_5km,numerical__nearby_wells_10km,numerical__nearby_wells_20km,numerical__well_age,numerical__construction_year_missing,status_group
0,69572,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5.0,109.0,1999.0,18.0,18.0,29.0,91.0,12.0,0.0,functional
1,8776,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2.0,280.0,2010.0,26.0,28.0,29.0,31.0,3.0,0.0,functional
2,34310,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,250.0,2009.0,16.0,16.0,18.0,106.0,4.0,0.0,functional
3,67743,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,63.0,58.0,1986.0,12.0,17.0,29.0,53.0,27.0,0.0,non functional
4,19728,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,NaN,8.0,18.0,47.0,156.0,NaN,1.0,functional
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59395,60739,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5.0,125.0,1999.0,59.0,179.0,574.0,1235.0,14.0,0.0,functional
59396,27263,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,56.0,1996.0,16.0,22.0,22.0,62.0,15.0,0.0,functional
59397,37057,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,7.0,0.0,NaN,2.0,3.0,19.0,204.0,NaN,1.0,functional
59398,31282,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,0.0,NaN,9.0,32.0,55.0,156.0,NaN,1.0,functional


In [ ]:
population_df.head()


In [ ]:
from pathlib import Path
from water_wells import TrainingPipeline, POPULATION_COLUMNS

root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent

# Reuse the shared pipeline implementation and persistent population cache.
population_data_pipeline = TrainingPipeline(
    population_raster=root / "tza_pop_2026_CN_1km_R2025A_UA_v1.tif",
    cache_dir=root / "data/processed/population",
    data_mode="auto",  # use "load" to forbid calculation on cache misses
)
training_df = population_data_pipeline.prepare_data(
    training_df,
    output_path=root / "data/processed/training_population_exploration.csv",
)
radius_columns = POPULATION_COLUMNS
training_df[["id", "population", *radius_columns.values()]].head()


In [41]:
pd.set_option('display.float_format', '{:.8f}'.format)

In [60]:
training_df.groupby(['latitude', 'longitude', 'population'])[['population_10km']].mean().reset_index().sort_values(by='population_10km').tail(600)

,latitude,longitude,population,population_10km
43632,-3.35341605,36.68387456,200,930228.31250000
43602,-3.35434536,36.69054976,200,930917.87500000
43485,-3.35811482,36.67331251,200,930927.18750000
43592,-3.35474838,36.69774981,200,931153.00000000
52563,-2.49645868,32.98478963,0,933041.25000000
...,...,...,...,...
21531,-6.89757912,39.21283461,250,2734865.25000000
21483,-6.90059583,39.21474861,80,2738851.00000000
21545,-6.89636320,39.21403003,300,2747130.50000000
21549,-6.89593450,39.21448429,148,2747452.25000000


In [35]:
population_df['population'].sum()

np.float32(7.1549624e+07)

In [ ]:
from pathlib import Path
from IPython.display import display
from water_wells import DataConfig, DataLoader, TrainingConfig, TrainingPipeline, ModelEvaluator

population_features = [
    "Population_1km", "Population_2km", "Population_3km",
    "population_5km", "population_10km", "population_20km",
]

# Population features are now generated and cached by the pipeline.
root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent
X_raw, y_model = DataLoader(DataConfig(data_dir=root)).load_training()
population_pipeline = TrainingPipeline(
    TrainingConfig(),
    population_raster=root / "tza_pop_2026_CN_1km_R2025A_UA_v1.tif",
    cache_dir=root / "data/processed/population",
    data_mode="auto",  # auto: reuse/write; load: cache only; write: refresh; off: no cache
)
# Prewarm both split subsets and save the enriched dataframe for exploration.
X_model = population_pipeline.prepare_data(
    X_raw, output_path=root / "data/processed/training_population.csv",
)
X_model.assign(status_group=y_model).to_csv(
    root / "data/processed/training_population_labels.csv", index=False,
)

X_train, X_valid, y_train, y_valid = population_pipeline.split(X_model, y_model)
population_pipeline.fit(X_train, y_train)

# Held-out predictions, class probabilities, and actual labels.
results_df = population_pipeline.predict_wells(X_valid)
results_df["actual_status"] = y_valid
results_df["correct"] = results_df["predicted_status"].eq(y_valid)
results_df[population_features] = X_valid[population_features]

evaluator = ModelEvaluator()
metrics_df = pd.DataFrame(
    [evaluator.metrics(y_valid, results_df["predicted_status"])],
    index=["with_population_features"],
)
population_importance_df = evaluator.feature_importance(population_pipeline)
population_importance_df = population_importance_df.loc[
    population_importance_df["feature"].isin(
        [f"numerical__{name}" for name in population_features]
    )
].reset_index(drop=True)
assert len(population_importance_df) == len(population_features)

display(metrics_df)
print(evaluator.classification_report(y_valid, results_df["predicted_status"]))
display(evaluator.confusion_matrix(y_valid, results_df["predicted_status"]))
display(population_importance_df)
display(results_df.head(10))
